In [3]:
# Instalação rápida das dependências
!pip install -q transformers pillow requests torch

import torch
import requests
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

# 1. Carrega o modelo e o processador do CLIP compacto
model_id = "openai/clip-vit-base-patch32"
print("⚡ Carregando modelo CLIP para avaliação multimodal...")
model = CLIPModel.from_pretrained(model_id)
processor = CLIPProcessor.from_pretrained(model_id)

# 2. Baixa uma imagem de teste (foto de um gato)
url = "https://images.unsplash.com/photo-1514888286974-6c03e2ca1dba?w=500"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")

# 3. Prompts para testar a qualidade do alinhamento (um correto e um incorreto)
prompt_correto = "a photo of a cute cat resting"
prompt_errado = "a photo of a sports car racing on the track"
prompts = [prompt_correto, prompt_errado]

# 4. Processa a imagem e os textos
inputs = processor(text=prompts, images=image, return_tensors="pt", padding=True)

with torch.no_grad():
    # Executa a inferência direta no modelo
    outputs = model(**inputs)

    # Obtém as probabilidades normalizadas via Softmax (CLIP Score / Alinhamento)
    logits_per_image = outputs.logits_per_image  # Similaridade entre imagem e textos
    probs = logits_per_image.softmax(dim=1).cpu().numpy()[0]

# 5. Exibe os resultados do alinhamento multimodal em aula
print("\n📊 RESULTADOS DA AVALIAÇÃO MULTIMODAL (CLIP Score):")
print("-" * 55)
print(f" Prompt 1 ('{prompt_correto}'): {probs[0] * 100:.2f}% de alinhamento")
print(f" Prompt 2 ('{prompt_errado}'): {probs[1] * 100:.2f}% de alinhamento")
print("-" * 55)
print("✅ Teste de avaliação concluído com sucesso!")

⚡ Carregando modelo CLIP para avaliação multimodal...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]


📊 RESULTADOS DA AVALIAÇÃO MULTIMODAL (CLIP Score):
-------------------------------------------------------
 Prompt 1 ('a photo of a cute cat resting'): 100.00% de alinhamento
 Prompt 2 ('a photo of a sports car racing on the track'): 0.00% de alinhamento
-------------------------------------------------------
✅ Teste de avaliação concluído com sucesso!
